# Gold Product Dimension

This notebook builds the `dim_products` Gold model by enriching Silver products with English category names.

**Grain:** One row per `product_id`.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths

In [0]:
SILVER_PRODUCTS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/products"
)

SILVER_CATEGORY_TRANSLATION_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/category_translation"
)

GOLD_DIM_PRODUCTS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/dim_products"
)

print(f"Products source: {SILVER_PRODUCTS_PATH}")
print(f"Category translation source: {SILVER_CATEGORY_TRANSLATION_PATH}")
print(f"Gold target: {GOLD_DIM_PRODUCTS_PATH}")

## 2. Read Silver inputs

In [0]:
silver_products_df = (
    spark.read
    .format("delta")
    .load(SILVER_PRODUCTS_PATH)
)

silver_category_translation_df = (
    spark.read
    .format("delta")
    .load(SILVER_CATEGORY_TRANSLATION_PATH)
)

silver_product_count = silver_products_df.count()
translation_count = silver_category_translation_df.count()

print(f"Silver product rows: {silver_product_count:,}")
print(f"Category translation rows: {translation_count:,}")

display(silver_products_df.limit(10))
display(silver_category_translation_df.limit(10))

## 3. Validate required columns

In [0]:
required_product_columns = {
    "product_id",
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume_cm3",
    "_silver_processed_at",
}

required_translation_columns = {
    "product_category_name",
    "product_category_name_english",
}

missing_product_columns = (
    required_product_columns - set(silver_products_df.columns)
)

missing_translation_columns = (
    required_translation_columns
    - set(silver_category_translation_df.columns)
)

if missing_product_columns:
    raise ValueError(
        "Silver products is missing required columns: "
        f"{sorted(missing_product_columns)}"
    )

if missing_translation_columns:
    raise ValueError(
        "Silver category translation is missing required columns: "
        f"{sorted(missing_translation_columns)}"
    )

print("Required column validation passed.")

## 4. Validate category translation grain

In [0]:
duplicate_translation_count = (
    silver_category_translation_df
    .groupBy("product_category_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_translation_key_count = (
    silver_category_translation_df
    .filter(F.col("product_category_name").isNull())
    .count()
)

if duplicate_translation_count > 0:
    raise ValueError(
        "Category translation contains "
        f"{duplicate_translation_count:,} duplicate category names."
    )

if null_translation_key_count > 0:
    raise ValueError(
        "Category translation contains "
        f"{null_translation_key_count:,} null category names."
    )

print("Category translation grain validation passed.")

## 5. Build product dimension

In [0]:
translation_df = (
    silver_category_translation_df
    .select(
        "product_category_name",
        "product_category_name_english",
    )
)

dim_products_df = (
    silver_products_df.alias("products")
    .join(
        translation_df.alias("translation"),
        on="product_category_name",
        how="left",
    )
    .select(
        F.col("products.product_id"),
        F.col("products.product_category_name"),
        F.coalesce(
            F.col("translation.product_category_name_english"),
            F.lit("unknown"),
        ).alias("product_category_name_english"),
        F.col("products.product_name_length"),
        F.col("products.product_description_length"),
        F.col("products.product_photos_qty"),
        F.col("products.product_weight_g"),
        F.col("products.product_length_cm"),
        F.col("products.product_height_cm"),
        F.col("products.product_width_cm"),
        F.col("products.product_volume_cm3"),
        F.col("products._silver_processed_at"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(dim_products_df.limit(10))

## 6. Validate product dimension

In [0]:
dim_product_count = dim_products_df.count()

duplicate_product_count = (
    dim_products_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_product_id_count = (
    dim_products_df
    .filter(F.col("product_id").isNull())
    .count()
)

null_english_category_count = (
    dim_products_df
    .filter(F.col("product_category_name_english").isNull())
    .count()
)

if dim_product_count == 0:
    raise ValueError("Product dimension is empty.")

if dim_product_count != silver_product_count:
    raise ValueError(
        "Product dimension row count does not match Silver products. "
        f"Silver: {silver_product_count:,}, "
        f"Gold: {dim_product_count:,}"
    )

if duplicate_product_count > 0:
    raise ValueError(
        f"Product dimension contains "
        f"{duplicate_product_count:,} duplicate product IDs."
    )

if null_product_id_count > 0:
    raise ValueError(
        f"Product dimension contains "
        f"{null_product_id_count:,} null product IDs."
    )

if null_english_category_count > 0:
    raise ValueError(
        "Product dimension contains "
        f"{null_english_category_count:,} null English category names."
    )

print(f"Product dimension rows: {dim_product_count:,}")
print("Product dimension grain validation passed.")

## 7. Inspect unmatched category translations

In [0]:
untranslated_categories_df = (
    dim_products_df
    .filter(
        F.col("product_category_name_english") == "untranslated"
    )
    .select("product_category_name")
    .distinct()
    .orderBy("product_category_name")
)

untranslated_category_count = untranslated_categories_df.count()

print(
    "Distinct categories without an English translation: "
    f"{untranslated_category_count:,}"
)

display(untranslated_categories_df)

## 8. Write product dimension to Gold

In [0]:
(
    dim_products_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_PRODUCTS_PATH)
)

print(f"Product dimension written to: {GOLD_DIM_PRODUCTS_PATH}")

## 9. Validate Gold output

In [0]:
written_dim_products_df = (
    spark.read
    .format("delta")
    .load(GOLD_DIM_PRODUCTS_PATH)
)

written_product_count = written_dim_products_df.count()

if written_product_count != dim_product_count:
    raise ValueError(
        "Gold product dimension write validation failed. "
        f"Expected: {dim_product_count:,}, "
        f"Written: {written_product_count:,}"
    )

print(f"Written product dimension rows: {written_product_count:,}")
print("Gold product dimension write validation passed.")

## 10. Inspect Gold product dimension

In [0]:
written_dim_products_df.printSchema()

display(
    written_dim_products_df
    .orderBy("product_id")
    .limit(10)
)